# 01 – PlantVillage: Exploration & Dataset Split

**Phase 1 notebook** – covers all exploration tasks from the plan:
1. Scan the dataset (count per class, corrupted files, resolutions)
2. Build the perceptual-hash-based leakage-safe split → `data/splits.csv`
3. Class distribution plot
4. Sample grid (one image per class)
5. Resolution distribution
6. Verify no hash group spans more than one split

> **Pre-requisite:** place the PlantVillage *color* subset at  
> `data/raw/color/<ClassName>/<image>.jpg`

In [ ]:
import sys, os
from pathlib import Path

# Make sure the project root is on sys.path
PROJECT_ROOT = Path('..').resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print('Project root:', PROJECT_ROOT)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from PIL import Image

from src.data import (
    scan_dataset, compute_phashes, assign_hash_groups,
    make_splits, build_splits, load_splits,
    DATA_RAW, DATA_DIR, SPLITS_CSV
)

sns.set_theme(style='whitegrid', palette='muted')
CHARTS_DIR = PROJECT_ROOT / 'data' / 'charts'
CHARTS_DIR.mkdir(parents=True, exist_ok=True)
print('Charts will be saved to:', CHARTS_DIR)

---
## 1. Scan the dataset

In [ ]:
df_raw = scan_dataset(DATA_RAW)

print(f"Total images   : {len(df_raw):,}")
print(f"Total classes  : {df_raw['class_name'].nunique()}")
print(f"Corrupted files: {df_raw['corrupted'].sum()}")
print()
df_raw.head(3)

In [ ]:
# Per-class image counts (sorted descending)
class_counts = (
    df_raw[~df_raw['corrupted']]
    .groupby('class_name')
    .size()
    .rename('count')
    .sort_values(ascending=False)
)
print(f"Most common class : {class_counts.idxmax()} ({class_counts.max():,} images)")
print(f"Least common class: {class_counts.idxmin()} ({class_counts.min():,} images)")
class_counts

In [ ]:
# Resolution summary
res_summary = df_raw[~df_raw['corrupted']][['width','height']].describe().T
print("Resolution statistics (pixels):")
res_summary

---
## 2. Build the leakage-safe split

If `splits.csv` already exists this cell just loads it; otherwise it runs  
the full hashing + union-find + GroupShuffleSplit pipeline.

> **This can take several minutes** on a cold run (hashing ~54 k images).

In [ ]:
if SPLITS_CSV.exists():
    print('splits.csv found – loading cached version.')
    df = load_splits(SPLITS_CSV)
else:
    print('Building splits from scratch (this may take a few minutes) …')
    df = build_splits(DATA_RAW, SPLITS_CSV)

print()
print(df.groupby('split').size().rename('images').to_string())

---
## 3. Leakage verification

No hash group should appear in more than one real split.

In [ ]:
real = df[df['split'].isin(['train', 'val', 'test'])]
splits_per_group = real.groupby('group_id')['split'].nunique()
leaking = (splits_per_group > 1).sum()

if leaking == 0:
    print('PASS: No hash group appears in more than one split.')
else:
    print(f'FAIL: {leaking} hash group(s) appear in more than one split!')

---
## Chart 1 – Class Distribution

In [ ]:
good = df[df['split'].isin(['train', 'val', 'test'])]
class_counts_split = (
    good.groupby(['class_name', 'split'])
    .size()
    .unstack(fill_value=0)
    [['train', 'val', 'test']]
)
# Sort by total
class_counts_split['_total'] = class_counts_split.sum(axis=1)
class_counts_split = class_counts_split.sort_values('_total').drop('_total', axis=1)

n_classes = len(class_counts_split)
fig, ax = plt.subplots(figsize=(10, max(8, n_classes * 0.28)))

class_counts_split.plot(
    kind='barh', stacked=True, ax=ax,
    color=['#4c72b0', '#dd8452', '#55a868'],
    edgecolor='none'
)
ax.set_xlabel('Number of images', fontsize=11)
ax.set_ylabel('')
ax.set_title('PlantVillage – Image Count per Class (colour version)', fontsize=13, fontweight='bold')
ax.legend(title='Split', loc='lower right')
plt.tight_layout()

out_path = CHARTS_DIR / 'class_distribution.png'
fig.savefig(out_path, dpi=150, bbox_inches='tight')
print(f'Saved: {out_path}')
plt.show()

---
## Chart 2 – Sample Grid (one image per class)

In [ ]:
# Pick one random train image per class
sample_rows = (
    good[good['split'] == 'train']
    .groupby('class_name')
    .apply(lambda g: g.sample(1, random_state=42))
    .reset_index(drop=True)
    .sort_values('class_name')
)

n_classes = len(sample_rows)
n_cols = 8
n_rows = (n_classes + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 2.2, n_rows * 2.4))
axes = axes.flatten()

for i, (_, row) in enumerate(sample_rows.iterrows()):
    img = Image.open(row['path']).convert('RGB')
    axes[i].imshow(img)
    # Short label: take last part after '___' if present
    label = row['class_name'].replace('___', '\n').replace('_', ' ')
    axes[i].set_title(label, fontsize=6, pad=2)
    axes[i].axis('off')

# Hide unused axes
for j in range(i + 1, len(axes)):
    axes[j].axis('off')

fig.suptitle('PlantVillage – One Sample per Class (colour)', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()

out_path = CHARTS_DIR / 'sample_grid.png'
fig.savefig(out_path, dpi=150, bbox_inches='tight')
print(f'Saved: {out_path}')
plt.show()

---
## Chart 3 – Resolution Distribution

In [ ]:
good_res = df_raw[~df_raw['corrupted']]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(good_res['width'],  bins=40, color='#4c72b0', edgecolor='white')
axes[0].set_xlabel('Width (px)',  fontsize=11)
axes[0].set_ylabel('Image count', fontsize=11)
axes[0].set_title('Image Width Distribution')

axes[1].hist(good_res['height'], bins=40, color='#dd8452', edgecolor='white')
axes[1].set_xlabel('Height (px)', fontsize=11)
axes[1].set_ylabel('Image count', fontsize=11)
axes[1].set_title('Image Height Distribution')

fig.suptitle('PlantVillage – Resolution Distribution', fontsize=13, fontweight='bold')
plt.tight_layout()

out_path = CHARTS_DIR / 'resolution_distribution.png'
fig.savefig(out_path, dpi=150, bbox_inches='tight')
print(f'Saved: {out_path}')
plt.show()

---
## Summary Report

In [ ]:
good = df[df['split'].isin(['train', 'val', 'test'])]

print('=' * 50)
print('PlantVillage (colour) – Phase 1 Summary')
print('=' * 50)
print(f"Total images (non-corrupted) : {len(good):,}")
print(f"Corrupted / excluded         : {df['corrupted'].sum()}")
print(f"Number of classes            : {good['class_name'].nunique()}")
print()
print('Split sizes:')
print(good.groupby('split').size().rename('images').to_string())
print()
print(f"Unique hash groups           : {good['group_id'].nunique():,}")
print(f"Leakage check                : {'PASS' if leaking == 0 else 'FAIL'}")
print()

# Imbalance ratio
per_class = good.groupby('class_name').size()
imbalance_ratio = per_class.max() / per_class.min()
print(f"Max/min class ratio          : {imbalance_ratio:.1f}x")
print(f"  -> Largest  : {per_class.idxmax()} ({per_class.max():,})")
print(f"  -> Smallest : {per_class.idxmin()} ({per_class.min():,})")
print()
print('Resolution:')
print(f"  Width  mean={good_res['width'].mean():.0f} px,  std={good_res['width'].std():.0f} px")
print(f"  Height mean={good_res['height'].mean():.0f} px, std={good_res['height'].std():.0f} px")
print('=' * 50)